# W03 — 파이썬 기초 ③ : 제어문 if · for · while · 리스트 컴프리헨션

> **Ping 데이터:** 심장질환 임상 데이터 (`heart.csv`) — Kaggle Heart Failure Prediction
> **Pong 데이터:** COVID-19 국가별 현황 (`compact.csv`) — Our World in Data
> **학습 목표:** 조건문(`if`)으로 분기, 반복문(`for`, `while`)으로 반복, `break`/`continue`로 흐름 제어,
> 리스트 컴프리헨션으로 간결한 리스트 생성을 익힌다.

→ W01에서 배운 변수, 숫자 연산, 문자열, f-string, 한 줄 if를 기반으로 한다.  
→ W02에서 배운 리스트를 적극 활용한다.

---
## 실습 데이터 로드

아래 셀을 **가장 먼저 실행**하라.
`pandas`는 W04에서 자세히 배운다. 지금은 **그냥 실행만** 하면 된다.

W03에서는 여러 환자·여러 국가 데이터를 **리스트**로 꺼내어 반복문에서 활용한다.

In [ ]:
# ┌─────────────────────────────────────────────────────────┐
# │  아래 코드는 지금 몰라도 된다. 그냥 실행만 하라.         │
# │  pandas와 데이터 로드는 W04에서 자세히 배운다.           │
# └─────────────────────────────────────────────────────────┘
import pandas as pd

BASE = "https://raw.githubusercontent.com/leina99-lab/classes/main/AI%ED%94%84%EB%A1%9C%EA%B7%B8%EB%9E%98%EB%B0%8D/data/"

# ── Ping: heart.csv 로드 (단일 환자 + 리스트) ─────────────────
df_heart = pd.read_csv(BASE + "heart.csv")

# 단일 환자 (1번 환자)
row          = df_heart.iloc[0]
age          = int(row["Age"])
sex          = str(row["Sex"])
resting_bp   = int(row["RestingBP"])
chol         = int(row["Cholesterol"])
max_hr       = int(row["MaxHR"])
oldpeak      = float(row["Oldpeak"])
heart_disease= int(row["HeartDisease"])
patient_id   = f"P-{df_heart.index[0]+1:03d}"

# 여러 환자 리스트 (앞 10명)
ages       = [int(x) for x in df_heart["Age"].iloc[:10]]
chols      = [int(x) for x in df_heart["Cholesterol"].iloc[:10]]
max_hrs    = [int(x) for x in df_heart["MaxHR"].iloc[:10]]
bp_list    = [int(x) for x in df_heart["RestingBP"].iloc[:10]]
diseases   = [int(x) for x in df_heart["HeartDisease"].iloc[:10]]
sex_list   = [str(x) for x in df_heart["Sex"].iloc[:10]]

print(f"[ heart.csv ] 총 {len(df_heart)}명 로드 완료")
print(f"단일 환자: {patient_id}, {age}세, 콜레스테롤 {chol}")
print(f"ages(10명)  = {ages}")
print(f"chols(10명) = {chols}")
print(f"max_hrs     = {max_hrs}")
print()

# ── Pong: compact.csv 로드 (한국 + 여러 국가 리스트) ──────────
def safe_int(x):   return int(x)   if pd.notna(x) else 0
def safe_float(x): return float(x) if pd.notna(x) else 0.0

df       = pd.read_csv(BASE + "compact.csv")
kr_df    = df[(df["code"] == "KOR") & (df["total_cases"].notna())]
kr       = kr_df.sort_values("date").iloc[-1]

country            = str(kr["country"])
date               = str(kr["date"])
total_cases        = safe_int(kr["total_cases"])
new_cases          = safe_int(kr["new_cases"])
total_deaths       = safe_int(kr["total_deaths"])
new_deaths         = safe_int(kr["new_deaths"])
total_vaccinations = safe_int(kr["total_vaccinations"])
people_vaccinated  = safe_int(kr["people_vaccinated"])
population         = safe_int(kr["population"])
gdp_per_capita     = safe_float(kr["gdp_per_capita"])
continent          = str(kr["continent"])

# 여러 국가 리스트 (G7 + 한국)
codes = ["KOR", "USA", "GBR", "FRA", "DEU", "JPN", "ITA", "CAN"]
country_names = []
country_cases = []
country_deaths = []
country_pops   = []
for c in codes:
    tmp = df[(df["code"]==c) & (df["total_cases"].notna())]
    if len(tmp) > 0:
        r = tmp.sort_values("date").iloc[-1]
        country_names.append(str(r["country"]))
        country_cases.append(safe_int(r["total_cases"]))
        country_deaths.append(safe_int(r["total_deaths"]))
        country_pops.append(safe_int(r["population"]))

print(f"[ compact.csv ] 총 {len(df)}행 로드 완료")
print(f"한국: {country}, {date}")
print(f"total_cases={total_cases:,}, total_deaths={total_deaths:,}")
print(f"국가 리스트: {country_names}")
print(f"확진자 리스트: {country_cases}")


---
# Section 1. 조건문 if 기초

## 개념 설명

**조건문**(Conditional Statement)이란, 조건의 참/거짓에 따라 실행 흐름을 분기하는 구문이다.

```python
if 조건식:            # 조건이 참이면
    실행할 문장        # 이 블록을 실행
```

→ 조건식 뒤에 **콜론**( `:` )을 반드시 붙인다.  
→ 실행할 문장은 **들여쓰기**(공백 4칸)로 구분한다.  
→ 파이썬은 **들여쓰기가 곧 코드 블록**이므로, 들여쓰기 오류에 주의하라.

```python
if 조건식:
    참일 때 실행
else:
    거짓일 때 실행
```

### Ping 1 — 환자 콜레스테롤이 200 초과인지 판별

In [ ]:
print("=== Ping 1: 기본 if문 ===")

if chol > 200:
    print(f"환자 {patient_id}: 콜레스테롤 {chol}mg/dL → 고콜레스테롤이다.")

if resting_bp >= 140:
    print(f"환자 {patient_id}: 혈압 {resting_bp}mmHg → 고혈압이다.")

if age >= 60:
    print(f"환자 {patient_id}: {age}세 → 고령 환자이다.")
else:
    print(f"환자 {patient_id}: {age}세 → 비고령 환자이다.")

if heart_disease == 1:
    print(f"환자 {patient_id}: 심장질환 있음")
else:
    print(f"환자 {patient_id}: 심장질환 없음")


### 개념 확인 — 조건문 if

다음 빈칸을 채워라.

1. `if` 뒤에 오는 조건식 끝에는 반드시 `___`를 붙여야 한다.
2. 조건이 참일 때 실행할 블록은 `___`(들여쓰기/내어쓰기)로 구분한다.
3. `if x > 0:`에서 `x = -5`이면 블록이 실행___(된다/안 된다).
4. `else`에는 별도의 조건식을 ___(적는다/적지 않는다).
5. 파이썬에서 들여쓰기 오류는 `___Error`를 발생시킨다.

<details>
<summary>→ 정답 보기</summary>

1. 콜론( `:` )  2. 들여쓰기  3. 안 된다  4. 적지 않는다  5. Indentation
</details>

### Pong 1 — COVID 데이터로 확산 상태를 판정하라.

<details>
<summary>→ 정답 보기</summary>

```python
print("=== Pong 1: 기본 if문 ===")
if new_cases > 10000:
    print(f"{country}: 신규확진 {new_cases:,}명 → 대규모 확산이다.")
else:
    print(f"{country}: 신규확진 {new_cases:,}명 → 관리 가능 수준이다.")

if total_deaths > 30000:
    print(f"누적 사망 {total_deaths:,}명 → 심각한 수준이다.")
else:
    print(f"누적 사망 {total_deaths:,}명 → 비교적 낮은 수준이다.")
```
</details>

In [ ]:
print("=== Pong 1: 기본 if문 ===")

if new_cases ___ 10000:
    print(f"{country}: 신규확진 {new_cases:,}명 → 대규모 확산이다.")
___:
    print(f"{country}: 신규확진 {new_cases:,}명 → 관리 가능 수준이다.")

if total_deaths ___ 30000:
    print(f"누적 사망 {total_deaths:,}명 → 심각한 수준이다.")
___:
    print(f"누적 사망 {total_deaths:,}명 → 비교적 낮은 수준이다.")


### Pong 2 — `gdp_per_capita`가 30000 이상이면 `'고소득 국가'`, 아니면 `'중·저소득 국가'`를 출력하라.

<details>
<summary>→ 정답 보기</summary>

```python
print("=== Pong 2: if-else ===")
if gdp_per_capita >= 30000:
    print(f'{country}: GDP ${gdp_per_capita:,.0f} → 고소득 국가')
else:
    print(f'{country}: GDP ${gdp_per_capita:,.0f} → 중·저소득 국가')
```
</details>

In [ ]:
print("=== Pong 2: if-else ===")

if gdp_per_capita ___ ___:
    print(f'{country}: GDP ${gdp_per_capita:,.0f} → 고소득 국가')
___:
    print(f'{country}: GDP ${gdp_per_capita:,.0f} → 중·저소득 국가')


---
# Section 2. if-elif-else — 다중 조건 분기

## 개념 설명

조건이 3개 이상일 때 `elif`(else if의 줄임)를 사용한다.

```python
if 조건1:
    조건1이 참일 때
elif 조건2:
    조건2가 참일 때
elif 조건3:
    조건3이 참일 때
else:
    모든 조건이 거짓일 때
```

→ 위에서부터 순서대로 검사하며, **처음으로 참인 조건**의 블록만 실행하고 나머지는 건너뛴다.

### Ping 2 — 환자 BMI 등급 판정 + 혈압 분류

In [ ]:
print("=== Ping 2: if-elif-else ===")

# BMI 예시 (키 170cm 가정)
height = 1.70
weight = 75
bmi = weight / (height ** 2)
print(f"BMI = {bmi:.1f}")

if bmi < 18.5:
    print("저체중이다.")
elif bmi < 23:
    print("정상 체중이다.")
elif bmi < 25:
    print("비만 전 단계이다.")
elif bmi < 30:
    print("1단계 비만이다.")
else:
    print("2단계 이상 비만이다.")

# 혈압 분류
print(f"\n혈압 {resting_bp}mmHg:")
if resting_bp < 90:
    print("저혈압이다.")
elif resting_bp < 120:
    print("정상이다.")
elif resting_bp < 140:
    print("고혈압 전 단계이다.")
else:
    print("고혈압이다.")


### Ping 3 — 학점 산출 (점수 기반)

In [ ]:
print("=== Ping 3: 학점 산출 ===")

score = 85

if score >= 90:
    grade = "A"
elif score >= 80:
    grade = "B"
elif score >= 70:
    grade = "C"
elif score >= 60:
    grade = "D"
else:
    grade = "F"

print(f"점수 {score} → 등급: {grade}")


### 개념 확인 — if-elif-else

다음 빈칸을 채워라.

1. `elif`는 `___` if의 줄임말이다.
2. if-elif-else에서 **처음으로 참인 조건**의 블록만 실행하고 나머지는 ___.
3. `score = 75`일 때 위 학점 코드의 결과 `grade`는 `'___'`이다.
4. `elif`는 원하는 만큼 ___(사용할 수 있다/1개만 가능하다).
5. `else`는 ___(필수이다/생략 가능하다).

<details>
<summary>→ 정답 보기</summary>

1. else  2. 건너뛴다  3. C  4. 사용할 수 있다  5. 생략 가능하다
</details>

### Pong 3 — COVID 사망률을 계산하고 위험도를 4단계로 분류하라.

<details>
<summary>→ 정답 보기</summary>

```python
print("=== Pong 3: 사망률 위험도 ===")
death_rate = total_deaths / total_cases * 100
print(f"사망률: {death_rate:.3f}%")

if death_rate >= 3:
    risk = '매우 위험'
elif death_rate >= 1:
    risk = '위험'
elif death_rate >= 0.5:
    risk = '주의'
else:
    risk = '안정'
print(f"위험도: {risk}")
```
</details>

In [ ]:
print("=== Pong 3: 사망률 위험도 ===")

death_rate = total_deaths / total_cases * 100
print(f"사망률: {death_rate:.3f}%")

if death_rate >= ___:
    risk = "매우 위험"
___ death_rate >= ___:
    risk = "위험"
___ death_rate >= ___:
    risk = "주의"
___:
    risk = "안정"

print(f"위험도: {risk}")


### Pong 4 — 접종률에 따라 집단면역 달성 여부를 판정하라. (70% 이상: 달성, 50~70%: 근접, 50% 미만: 미달)

<details>
<summary>→ 정답 보기</summary>

```python
print("=== Pong 4: 접종률 판정 ===")
vax_rate = people_vaccinated / population * 100
print(f"접종률: {vax_rate:.1f}%")

if vax_rate >= 70:
    status = '집단면역 달성'
elif vax_rate >= 50:
    status = '집단면역 근접'
else:
    status = '집단면역 미달'
print(f"상태: {status}")
```
</details>

In [ ]:
print("=== Pong 4: 접종률 판정 ===")

vax_rate = ___ / ___ * 100
print(f"접종률: {vax_rate:.1f}%")

if vax_rate ___ 70:
    status = "집단면역 달성"
___ vax_rate ___ 50:
    status = "집단면역 근접"
___:
    status = "집단면역 미달"

print(f"상태: {status}")


---
# Section 3. 반복문 for 기초와 range()

## 개념 설명

**반복문**(Loop)이란, 특정 작업을 여러 번 되풀이하는 구문이다.  
`for`문은 **반복 횟수가 정해져 있을 때** 사용한다.

```python
for 변수 in range(n):     # 0부터 n-1까지 반복
    반복할 문장
```

| `range()` 형태 | 의미 |
|:---|:---|
| `range(5)` | 0, 1, 2, 3, 4 |
| `range(2, 5)` | 2, 3, 4 |
| `range(0, 10, 2)` | 0, 2, 4, 6, 8 |
| `range(10, 0, -1)` | 10, 9, 8, ..., 1 |

### Ping 4 — 10명 환자의 나이를 하나씩 출력 (for + range)

In [ ]:
print("=== Ping 4: for + range ===")

print("--- range(len(ages)) 방식 ---")
for i in range(len(ages)):
    print(f"환자 {i+1}: {ages[i]}세")

print("\n--- 직접 순회 방식 ---")
for a in ages:
    print(f"나이: {a}세")


### Ping 5 — 10명 환자의 나이 합계·평균 계산

In [ ]:
print("=== Ping 5: 누적 합 ===")

total = 0
for a in ages:
    total = total + a

avg = total / len(ages)
print(f"나이 합계: {total}")
print(f"나이 평균: {avg:.1f}세")
print(f"환자 수: {len(ages)}명")


### 개념 확인 — for문과 range()

다음 빈칸을 채워라.

1. `range(5)`는 `___`부터 `___`까지 생성한다.
2. `range(2, 8)`은 `___`부터 `___`까지 생성한다.
3. `range(0, 10, 3)`은 `___`을 생성한다.
4. 리스트 `ls`의 길이만큼 반복하려면 `for i in range(___)`이라 쓴다.
5. `for a in ages:`에서 변수 `a`에는 리스트의 요소가 ___대로 대입된다.

<details>
<summary>→ 정답 보기</summary>

1. 0부터 4까지  2. 2부터 7까지  3. 0, 3, 6, 9  4. `len(ls)`  5. 차례
</details>

### Pong 5 — for문으로 여러 국가의 확진자를 출력하라.

<details>
<summary>→ 정답 보기</summary>

```python
print("=== Pong 5: 국가별 확진자 ===")
for i in range(len(country_names)):
    print(f'{country_names[i]}: 확진 {country_cases[i]:,}명')
```
</details>

In [ ]:
print("=== Pong 5: 국가별 확진자 ===")

for i in range(___(country_names)):
    print(f'{country_names[___]}: 확진 {country_cases[___]:,}명')


### Pong 6 — for문으로 국가별 사망률(total_deaths/total_cases*100)을 계산하고 출력하라.

<details>
<summary>→ 정답 보기</summary>

```python
print("=== Pong 6: 국가별 사망률 ===")
for i in range(len(country_names)):
    if country_cases[i] > 0:
        rate = country_deaths[i] / country_cases[i] * 100
        print(f'{country_names[i]}: 사망률 {rate:.3f}%')
```
</details>

In [ ]:
print("=== Pong 6: 국가별 사망률 ===")

for i in range(len(country_names)):
    if country_cases[i] ___ 0:
        rate = country_deaths[___] / country_cases[___] * 100
        print(f'{country_names[i]}: 사망률 {rate:.3f}%')


---
# Section 4. for + if 조합 — 조건부 필터링

## 개념 설명

for문 안에 if문을 넣으면 **조건을 만족하는 요소만** 처리할 수 있다.

```python
result = []
for x in 리스트:
    if 조건:
        result.append(x)
```

### Ping 6 — 콜레스테롤 200 초과 환자만 필터링

In [ ]:
print("=== Ping 6: for + if 필터링 ===")

high_chol = []
for c in chols:
    if c > 200:
        high_chol.append(c)

print(f"전체 환자: {chols}")
print(f"고콜레스테롤(>200): {high_chol}")
print(f"고콜레스테롤 환자 수: {len(high_chol)}명")


### Ping 7 — 심장질환이 있는(1) 환자의 나이만 추출

In [ ]:
print("=== Ping 7: 조건부 추출 ===")

disease_ages = []
for i in range(len(diseases)):
    if diseases[i] == 1:
        disease_ages.append(ages[i])

print(f"심장질환 환자의 나이: {disease_ages}")
print(f"평균 나이: {sum(disease_ages)/len(disease_ages):.1f}세" if disease_ages else "해당 환자 없음")


### Pong 7 — 국가별 확진자가 1000만 이상인 국가만 리스트에 담아 출력하라.

<details>
<summary>→ 정답 보기</summary>

```python
print("=== Pong 7: 확진자 1000만 이상 ===")
big_countries = []
for i in range(len(country_names)):
    if country_cases[i] >= 10_000_000:
        big_countries.append(country_names[i])
print(f'확진 1000만 이상: {big_countries}')
```
</details>

In [ ]:
print("=== Pong 7: 확진자 1000만 이상 ===")

big_countries = []
for i in range(len(country_names)):
    if country_cases[i] >= ___:
        big_countries.___(country_names[i])

print(f"확진 1000만 이상: {big_countries}")


### Pong 8 — 국가별 인구 대비 확진률(%)을 계산하고, 10% 이상인 국가만 출력하라.

<details>
<summary>→ 정답 보기</summary>

```python
print("=== Pong 8: 확진률 10% 이상 ===")
for i in range(len(country_names)):
    rate = country_cases[i] / country_pops[i] * 100
    if rate >= 10:
        print(f'{country_names[i]}: 확진률 {rate:.1f}%')
```
</details>

In [ ]:
print("=== Pong 8: 확진률 10% 이상 ===")

for i in range(len(country_names)):
    rate = country_cases[___] / country_pops[___] * 100
    if rate ___ 10:
        print(f'{country_names[i]}: 확진률 {rate:.1f}%')


---
# Section 5. while문, break, continue

## 개념 설명

`while`문은 **조건이 참인 동안** 계속 반복한다.  
`for`문과 달리 **반복 횟수를 미리 모를 때** 적합하다.

```python
i = 0          # 초기값
while i < 5:   # 조건이 참인 동안
    print(i)   # 실행
    i += 1     # 변수 갱신 (없으면 무한루프!)
```

| 키워드 | 기능 |
|:---:|:---|
| `break` | 반복문을 **즉시 종료** |
| `continue` | 현재 반복의 나머지를 **건너뛰고** 다음 반복으로 |

### Ping 8 — while로 환자 나이 누적 합이 300 넘으면 중단

In [ ]:
print("=== Ping 8: while + break ===")

total = 0
count = 0
i = 0
while i < len(ages):
    total += ages[i]
    count += 1
    print(f"  환자{count}: {ages[i]}세 추가 → 누적: {total}")
    if total > 300:
        print(f"  → 누적 나이 {total}세 > 300. 중단!")
        break
    i += 1

print(f"처리된 환자 수: {count}명")


### Ping 9 — continue로 심장질환 없는 환자 건너뛰기

In [ ]:
print("=== Ping 9: continue ===")

for i in range(len(diseases)):
    if diseases[i] == 0:    # 심장질환 없으면
        continue            # 건너뛴다
    print(f"환자 {i+1}: {ages[i]}세, 심장질환 있음, 콜레스테롤 {chols[i]}")


### 개념 확인 — while, break, continue

다음 빈칸을 채워라.

1. `while`문은 조건이 ___인 동안 반복한다.
2. while문에서 변수 갱신이 없으면 ___루프에 빠진다.
3. `break`는 반복문을 즉시 ___한다.
4. `continue`는 현재 반복의 나머지를 ___ 다음 반복으로 넘어간다.
5. `break`와 달리 `continue`는 반복문 자체를 ___(종료한다/종료하지 않는다).

<details>
<summary>→ 정답 보기</summary>

1. 참  2. 무한  3. 종료  4. 건너뛰고  5. 종료하지 않는다
</details>

### Pong 9 — while문으로 `country_cases` 리스트를 순회하다가 확진자 5000만 이상인 국가를 만나면 중단하라.

<details>
<summary>→ 정답 보기</summary>

```python
print("=== Pong 9: while + break ===")
i = 0
while i < len(country_names):
    print(f'{country_names[i]}: {country_cases[i]:,}명')
    if country_cases[i] >= 50_000_000:
        print('→ 5000만 이상! 중단.')
        break
    i += 1
```
</details>

In [ ]:
print("=== Pong 9: while + break ===")

i = 0
while i < len(country_names):
    print(f'{country_names[i]}: {country_cases[i]:,}명')
    if country_cases[i] >= ___:
        print("→ 5000만 이상! 중단.")
        ___
    i += 1


### Pong 10 — for문과 continue를 사용하여 사망자가 0인 국가를 건너뛰고 나머지만 출력하라.

<details>
<summary>→ 정답 보기</summary>

```python
print("=== Pong 10: continue ===")
for i in range(len(country_names)):
    if country_deaths[i] == 0:
        continue
    rate = country_deaths[i] / country_cases[i] * 100
    print(f'{country_names[i]}: 사망률 {rate:.3f}%')
```
</details>

In [ ]:
print("=== Pong 10: continue ===")

for i in range(len(country_names)):
    if country_deaths[i] ___ 0:
        ___
    rate = country_deaths[i] / country_cases[i] * 100
    print(f'{country_names[i]}: 사망률 {rate:.3f}%')


---
# Section 6. 중첩 for문

## 개념 설명

for문 안에 for문을 넣으면 **이중 반복**이 된다.  
바깥 for문이 한 번 실행될 때마다, 안쪽 for문은 **처음부터 끝까지** 모두 실행된다.

```python
for i in range(3):       # 바깥 (0, 1, 2)
    for j in range(4):   # 안쪽 (0, 1, 2, 3)
        print(i, j)      # 총 3 × 4 = 12번 실행
```

### Ping 10 — 구구단 출력

In [ ]:
print("=== Ping 10: 구구단 ===")

for i in range(2, 10):
    for j in range(1, 10):
        print(f"{i}x{j}={i*j}", end="\t")
    print()  # 단 끝나면 줄바꿈


### Ping 11 — 환자별로 3가지 수치(나이, 콜레스테롤, 심박수) 출력

In [ ]:
print("=== Ping 11: 중첩 for ===")

labels  = ["나이", "콜레스테롤", "심박수"]
data    = [ages, chols, max_hrs]

for i in range(3):  # 환자 3명만
    print(f"--- 환자 {i+1} ---")
    for j in range(len(labels)):
        print(f"  {labels[j]}: {data[j][i]}")


### Pong 11 — 이중 for문을 사용하여 국가명의 각 글자를 한 줄에 출력하라 (앞 3개국만).

<details>
<summary>→ 정답 보기</summary>

```python
print("=== Pong 11: 중첩 for ===")
for i in range(3):
    name = country_names[i]
    print(f'{name}:', end=' ')
    for ch in name:
        print(ch, end='-')
    print()
```
</details>

In [ ]:
print("=== Pong 11: 중첩 for ===")

for i in range(___):
    name = country_names[i]
    print(f'{name}:', end=' ')
    for ch in ___:
        print(ch, end='-')
    print()


---
# Section 7. 리스트 컴프리헨션(List Comprehension)

## 개념 설명

**리스트 컴프리헨션**이란, for문과 조건문을 **한 줄로 압축**하여 리스트를 만드는 파이썬 고유의 문법이다.  
기존 for + append 방식보다 **간결하고 빠르다**.

### 기본 구조
```python
# for문 방식
result = []
for x in 리스트:
    result.append(표현식)

# 컴프리헨션 (동일한 결과)
result = [표현식 for x in 리스트]
```

### 조건 필터링
```python
# for문 방식
result = []
for x in 리스트:
    if 조건:
        result.append(x)

# 컴프리헨션 (동일한 결과)
result = [x for x in 리스트 if 조건]
```

### 조건부 값 변환 (if-else)
```python
result = [참값 if 조건 else 거짓값 for x in 리스트]
```

→ **필터링** if는 for **뒤**에, **값 변환** if-else는 for **앞**에 온다는 점에 주의하라!

### Ping 12 — for문 → 컴프리헨션 변환 비교

In [ ]:
print("=== Ping 12: 컴프리헨션 기본 ===")

# for문 방식: 나이 제곱 리스트
age_sq_for = []
for a in ages:
    age_sq_for.append(a ** 2)
print(f"for문:      {age_sq_for}")

# 컴프리헨션: 동일한 결과
age_sq_comp = [a ** 2 for a in ages]
print(f"컴프리헨션: {age_sq_comp}")
print(f"동일한가? {age_sq_for == age_sq_comp}")


### Ping 13 — 조건 필터링 컴프리헨션

In [ ]:
print("=== Ping 13: 조건 필터링 ===")

# for문 방식: 콜레스테롤 200 초과
high_for = []
for c in chols:
    if c > 200:
        high_for.append(c)
print(f"for문:      {high_for}")

# 컴프리헨션: 동일한 결과
high_comp = [c for c in chols if c > 200]
print(f"컴프리헨션: {high_comp}")


### Ping 14 — 값 변환 컴프리헨션 (if-else)

In [ ]:
print("=== Ping 14: 값 변환 ===")

# 각 환자의 심장질환 유무를 문자열로 변환
labels = ["있음" if d == 1 else "없음" for d in diseases]
print(f"질환 유무: {labels}")

# 혈압 정상/비정상 분류
bp_labels = ["고혈압" if bp >= 140 else "정상" for bp in bp_list]
print(f"혈압 분류: {bp_labels}")

# 나이 → 고령/비고령
age_labels = ["고령" if a >= 60 else "비고령" for a in ages]
print(f"연령 분류: {age_labels}")


### 개념 확인 — 리스트 컴프리헨션

다음 빈칸을 채워라.

1. `[x**2 for x in range(5)]`의 결과는 `___`이다.
2. `[x for x in range(10) if x % 2 == 0]`의 결과는 `___`이다.
3. 필터링용 `if`는 for의 ___(앞/뒤)에, 값 변환용 `if-else`는 for의 ___(앞/뒤)에 온다.
4. 컴프리헨션 `[c for c in chols if c > 200]`을 for문으로 풀면 ___ 줄이 필요하다.
5. `['짝수' if x%2==0 else '홀수' for x in range(4)]`의 결과는 `___`이다.

<details>
<summary>→ 정답 보기</summary>

1. `[0, 1, 4, 9, 16]`  2. `[0, 2, 4, 6, 8]`  3. 뒤, 앞  4. 4줄 (빈리스트, for, if, append)  5. `['짝수', '홀수', '짝수', '홀수']`
</details>

### Pong 12 — 컴프리헨션으로 국가별 사망률 리스트를 만들어라.

<details>
<summary>→ 정답 보기</summary>

```python
print("=== Pong 12: 사망률 컴프리헨션 ===")
death_rates = [country_deaths[i]/country_cases[i]*100 for i in range(len(country_names))]
for i in range(len(country_names)):
    print(f'{country_names[i]}: {death_rates[i]:.3f}%')
```
</details>

In [ ]:
print("=== Pong 12: 사망률 컴프리헨션 ===")

death_rates = [country_deaths[i]/country_cases[i]*100 for i in range(___(country_names))]

for i in range(len(country_names)):
    print(f'{country_names[i]}: {death_rates[i]:.3f}%')


### Pong 13 — 컴프리헨션으로 확진자 1000만 이상인 국가 이름만 추출하라.

<details>
<summary>→ 정답 보기</summary>

```python
print("=== Pong 13: 필터링 컴프리헨션 ===")
big = [country_names[i] for i in range(len(country_names)) if country_cases[i] >= 10_000_000]
print(f'확진 1000만 이상: {big}')
```
</details>

In [ ]:
print("=== Pong 13: 필터링 컴프리헨션 ===")

big = [country_names[i] for i in range(len(country_names)) if country_cases[i] ___ ___]
print(f"확진 1000만 이상: {big}")


### Pong 14 — if-else 컴프리헨션으로 각 국가의 위험도를 `'위험'`(사망률 0.5% 이상) / `'안정'`으로 분류하라.

<details>
<summary>→ 정답 보기</summary>

```python
print("=== Pong 14: 값 변환 컴프리헨션 ===")
risk_labels = [
    '위험' if country_deaths[i]/country_cases[i]*100 >= 0.5 else '안정'
    for i in range(len(country_names))
]
for i in range(len(country_names)):
    print(f'{country_names[i]}: {risk_labels[i]}')
```
</details>

In [ ]:
print("=== Pong 14: 값 변환 컴프리헨션 ===")

risk_labels = [
    ___ if country_deaths[i]/country_cases[i]*100 >= 0.5 ___ ___
    for i in range(len(country_names))
]

for i in range(len(country_names)):
    print(f'{country_names[i]}: {risk_labels[i]}')


---
# Section 8. 종합 실습

if, for, while, break/continue, 리스트 컴프리헨션을 모두 활용한다.

### Ping 15 — 환자 종합 위험도 보고서 (모든 개념 활용)

In [ ]:
print("=== Ping 15: 환자 종합 위험도 보고서 ===")
print("=" * 50)

# 위험 요소 카운트
risk_count = 0
risk_items = []

for i in range(len(ages)):
    risk_count = 0
    risk_items = []
    
    if ages[i] >= 60:
        risk_count += 1
        risk_items.append("고령")
    if chols[i] > 200:
        risk_count += 1
        risk_items.append("고콜레스테롤")
    if bp_list[i] >= 140:
        risk_count += 1
        risk_items.append("고혈압")
    
    if risk_count >= 2:
        level = "고위험"
    elif risk_count == 1:
        level = "주의"
    else:
        level = "정상"
    
    print(f"환자{i+1} ({ages[i]}세): [{level}] 위험요소={risk_items}")

print("=" * 50)

# 컴프리헨션으로 고위험 환자 수 계산
high_risk_count = len([1 for i in range(len(ages))
                       if (ages[i]>=60) + (chols[i]>200) + (bp_list[i]>=140) >= 2])
print(f"고위험 환자 수: {high_risk_count}명 / {len(ages)}명")


### Pong 15 — 배운 모든 것을 활용하여 국가별 종합 보고서를 완성하라.

<details>
<summary>→ 정답 보기</summary>

```python
print("=== Pong 15: 국가별 종합 보고서 ===")
print("=" * 55)

for i in range(len(country_names)):
    d_rate = country_deaths[i] / country_cases[i] * 100
    c_rate = country_cases[i] / country_pops[i] * 100
    
    if d_rate >= 1:
        d_level = '위험'
    elif d_rate >= 0.5:
        d_level = '주의'
    else:
        d_level = '안정'
    
    print(f'{country_names[i]:15s} | 확진률 {c_rate:.1f}% | '
          f'사망률 {d_rate:.3f}% [{d_level}]')

print("=" * 55)

safe_countries = [country_names[i] for i in range(len(country_names))
                  if country_deaths[i]/country_cases[i]*100 < 0.5]
print(f'사망률 0.5% 미만 국가: {safe_countries}')
```
</details>

In [ ]:
print("=== Pong 15: 국가별 종합 보고서 ===")
print("=" * 55)

for i in range(len(country_names)):
    d_rate = country_deaths[i] / country_cases[i] * 100
    c_rate = country_cases[i] / country_pops[i] * 100
    
    if d_rate ___ 1:
        d_level = "위험"
    ___ d_rate ___ 0.5:
        d_level = "주의"
    ___:
        d_level = "안정"
    
    print(f'{country_names[i]:15s} | 확진률 {c_rate:.1f}% | '
          f'사망률 {d_rate:.3f}% [{d_level}]')

print("=" * 55)

# 컴프리헨션: 사망률 0.5% 미만 국가만 추출
safe_countries = [country_names[i] for i in range(len(country_names))
                  if country_deaths[i]/country_cases[i]*100 ___ ___]
print(f"사망률 0.5% 미만 국가: {safe_countries}")


---
# Section 9. 연습문제

스스로 풀어보고 정답을 확인하라.

### 연습 1 — for문으로 `ages` 리스트에서 50세 이상인 나이만 출력하라.

<details>
<summary>→ 정답 보기</summary>

```python
for a in ages:
    if a >= 50:
        print(a)
```
</details>

In [ ]:
# 여기에 직접 풀어보라


### 연습 2 — `max_hrs` 리스트의 최솟값을 for문으로 구하라. (min() 함수 사용 금지)

<details>
<summary>→ 정답 보기</summary>

```python
min_val = max_hrs[0]
for h in max_hrs:
    if h < min_val:
        min_val = h
print(f'최솟값: {min_val}')
```
</details>

In [ ]:
# 여기에 직접 풀어보라


### 연습 3 — 리스트 컴프리헨션으로 `bp_list`에서 120 미만인 값만 추출하라.

<details>
<summary>→ 정답 보기</summary>

```python
low_bp = [bp for bp in bp_list if bp < 120]
print(low_bp)
```
</details>

In [ ]:
# 여기에 직접 풀어보라


### 연습 4 — `country_cases`에서 가장 큰 값과 해당 국가명을 for문으로 찾아라.

<details>
<summary>→ 정답 보기</summary>

```python
max_c = 0
max_name = ''
for i in range(len(country_names)):
    if country_cases[i] > max_c:
        max_c = country_cases[i]
        max_name = country_names[i]
print(f'{max_name}: {max_c:,}명')
```
</details>

In [ ]:
# 여기에 직접 풀어보라


### 연습 5 — while문을 사용하여 1부터 더해 나가다가 합이 `total_deaths`를 넘는 순간 멈추고, 그때의 n을 출력하라.

<details>
<summary>→ 정답 보기</summary>

```python
s = 0
n = 0
while s <= total_deaths:
    n += 1
    s += n
print(f'n = {n}, 합 = {s:,}')
```
</details>

In [ ]:
# 여기에 직접 풀어보라


### 연습 6 — 리스트 컴프리헨션으로 `ages`의 각 나이를 `'고령'`(60 이상) / `'중년'`(40 이상) / `'청년'`으로 분류하라.

<details>
<summary>→ 정답 보기</summary>

```python
labels = ['고령' if a>=60 else ('중년' if a>=40 else '청년') for a in ages]
print(labels)
```
</details>

In [ ]:
# 여기에 직접 풀어보라


### 연습 7 — for문으로 `chols`와 `max_hrs`를 동시에 순회하며 `f'콜레스테롤 {c}, 심박수 {h}'` 형태로 출력하라.

<details>
<summary>→ 정답 보기</summary>

```python
for i in range(len(chols)):
    print(f'콜레스테롤 {chols[i]}, 심박수 {max_hrs[i]}')
```
</details>

In [ ]:
# 여기에 직접 풀어보라


### 연습 8 — for문과 continue를 사용하여 `sex_list`에서 `'M'`인 환자만 나이를 출력하라.

<details>
<summary>→ 정답 보기</summary>

```python
for i in range(len(sex_list)):
    if sex_list[i] != 'M':
        continue
    print(f'남성 환자: {ages[i]}세')
```
</details>

In [ ]:
# 여기에 직접 풀어보라


### 연습 9 — 구구단 중 결과가 30 이상인 것만 리스트 컴프리헨션으로 `'2x5=10'` 형태의 문자열 리스트로 만들어라.

<details>
<summary>→ 정답 보기</summary>

```python
result = [f'{i}x{j}={i*j}' for i in range(2,10) for j in range(1,10) if i*j >= 30]
print(result[:10])  # 앞 10개
```
</details>

In [ ]:
# 여기에 직접 풀어보라


### 연습 10 — 리스트 컴프리헨션으로 `country_names`와 `country_cases`를 조합하여 `'국가: N명'` 형태의 문자열 리스트를 만들어라.

<details>
<summary>→ 정답 보기</summary>

```python
report = [f'{country_names[i]}: {country_cases[i]:,}명' for i in range(len(country_names))]
for r in report:
    print(r)
```
</details>

In [ ]:
# 여기에 직접 풀어보라
